In [1]:
#Install required libraries
!pip install -q transformers faiss-cpu langchain PyPDF2 sentence-transformers evaluate
!pip install -q -U langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.1 MB/s eta 0:00:00


In [2]:
# Imports and setup
from PyPDF2 import PdfReader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.prompts import PromptTemplate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import evaluate
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
# Load PDF and extract text
def load_pdf(path):
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)

pdf_path = "/content/test.pdf"
pdf_text = load_pdf(pdf_path)
print(f"Extracted {len(pdf_text)} characters from PDF")

Extracted 133653 characters from PDF


In [4]:
# Chunk the text
splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""],
    strip_whitespace=True
)
chunks = splitter.split_text(pdf_text)
print(f"Created {len(chunks)}")

Created 337


In [5]:
chunks[13]

'II. D ISTRIBUTED REPRESENTATION\nStatistical NLP has emerged as the primary option for modeling complex natural language tasks. However, in its beginning,\nit often used to suffer from the notorious curse of dimensionality while learning joint probability functions of language models.\nThis led to the motivation of learning distributed representations of words existing in low-dimensional space [7].\nA. Word Embeddings\nDistributional vectors or word embeddings (Fig. 2) essentially follow the distributional hypothesis, according to which words'

In [6]:
emb_model_name = "sentence-transformers/all-mpnet-base-v2"
embeddings = HuggingFaceEmbeddings(model_name=emb_model_name)

# This builds embeddings & FAISS index under the hood
vectorstore = FAISS.from_texts(chunks, embeddings)
print("Vectorstore ready!")

<ipython-input-6-3887769113>:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=emb_model_name)
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Vectorstore ready!


In [7]:
REFUSAL = "I do not have context for that to answer"

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "You are an AI assistant named ChatPDF that answers *only* from the provided PDF context below.\n\n"
        "{context}\n\n"
        "Question: {question}\n\n"
        f"If the answer cannot be found in the context, reply exactly: \"{REFUSAL}\"."
    )
)

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
gen_pipe = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_length=512,
    device_map="auto"
)


Device set to use cuda:0


In [8]:
def chat_with_pdf(question, top_k=8, use_k=4):
    # retrieve
    docs = vectorstore.similarity_search(question, k=top_k)
    context = "\n\n".join(d.page_content for d in docs[:use_k])
    if not context.strip():
        return REFUSAL

    # format & generate
    prompt_text = prompt.format(context=context, question=question)
    out = gen_pipe(prompt_text, do_sample=False)[0]["generated_text"]

    # extract after “Answer:” if present
    answer = out.split("Answer:")[-1].strip()
    if len(answer) < 5 or answer.lower().startswith(question.lower()):
        return REFUSAL
    return answer


In [12]:
for q in [
    "What is the main contribution of the paper?",
    "Which method is proposed?",
    "When was it published?",
    "Who invented the Transformer?",
    "Who is the prime minister of Nepal?"
]:
    print(f"Q: {q}\nA: {chat_with_pdf(q)}\n{'-'*60}")


Q: What is the main contribution of the paper?
A: A unified architecture for natural language processing: Deep neural networks with multitask learning
------------------------------------------------------------
Q: Which method is proposed?
A: I do not have context for that to answer
------------------------------------------------------------


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Q: When was it published?
A: 2015, pp. 167–176. [66] O. Abdel-Hamid, A.-r. Mohamed, H. Jiang, L. Deng, G. Penn, and D. Yu, “Convolutional neural networks for speech and recursive neural networks, as well as their use in various NLP tasks; following, Section VI lists recent applications of reinforcement learning in NLP and new developments in unsupervised sentence representation learning; later, Section VII means authors contributed equally Corresponding author (e-mail: cambria@ntu.edu.sg) 1We intend to update this article with time as and when significant advances are proposed and used by the communityarXiv:1708.02709v8 [cs.CL] 25 Nov 2018 2 Fig. 1: Percentage of deep learning papers in ACL, EMNLP, EACL, NAACL over the last 6 years (long papers). Kim [50] and Kalchbrenner et al. , “Event extraction via dynamic multi-pooling convolutional neural networks.” in ACL (1) , 2015, pp. 167–176. [66] O. Abdel-Hamid, A.-r. Mohamed, H. Jiang, L. Deng, G. Penn, and D. Yu, “Convolutional neural net

In [10]:
test_set = [
    {"q": "What problem does the PDF address?",
     "a": "It addresses the limitations of recurrent and convolutional sequence transduction models by proposing an attention-only architecture."},
    {"q": "Which method is proposed?",
     "a": "The Transformer, a neural network architecture based solely on attention mechanisms, dispensing with recurrence and convolutions."},
    {"q": "When was it published?",
     "a": "June 12, 2017"}
]

refs, preds = [], []
for item in test_set:
    pred = chat_with_pdf(item["q"])
    print(f"Q: {item['q']}\nRef: {item['a']}\nGen: {pred}\n{'-'*40}")
    refs.append(item["a"])
    preds.append(pred)


Q: What problem does the PDF address?
Ref: It addresses the limitations of recurrent and convolutional sequence transduction models by proposing an attention-only architecture.
Gen: The problem arises also if the input is long or very information-rich and selective encoding is not possible. For example, the task of text summarization can be cast as a sequence-to-sequence learning problem, where the input is the original text and the output is the condensed version. In tasks such as text summarization and machine translation, certain alignment exists between the input text and the output text, which means that each token generation step is highly related to a certain part of the input text. This intuition inspires the Fig. 13: Neural-image QA (Figure source: Malinowski et al. [101]) 15 attention mechanism. This mechanism attempts to ease the above problems by allowing the decoder to refer back to the input a max-pooling strategy, c=maxfcg, which subsamples the input typically by applyin

In [11]:
# simple semantic similarity
embedder = SentenceTransformer("all-MiniLM-L6-v2")
ref_embeds = embedder.encode(refs)
pred_embeds = embedder.encode(preds)
scores = np.diag(cosine_similarity(pred_embeds, ref_embeds))

for q, score in zip([t["q"] for t in test_set], scores):
    print(f"Similarity ({score:.3f}) for: {q}")


Similarity (0.531) for: What problem does the PDF address?
Similarity (0.124) for: Which method is proposed?
Similarity (0.155) for: When was it published?
